# Dataset 3 – Rice Grains
**Implementation:** Akshit (algorithms/akshit/)  
**Algorithms:** TBD  
**Evaluation:** Stratified 10-fold CV · Accuracy · Macro F1

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['git', 'clone', f'https://github.com/vishalhanuman14/classical-ml-benchmark.git'], check=True)
    os.chdir('classical-ml-benchmark')

if '.' not in sys.path:
    sys.path.insert(0, '.')


# Dataset 3 – Rice Grains
**Implementation:** Akshit (algorithms/akshit/) for RF/DT · Vishal (algorithms/vishal/) for Neural Network  
**Algorithms:** Random Forest · Neural Network  
**Evaluation:** Stratified 10-fold CV · Accuracy · Macro F1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
sys.path.insert(0, '.')

from algorithms.vishal import random_forest, neural_network, utils

np.random.seed(42)
os.makedirs('results', exist_ok=True)

df = pd.read_csv('data/rice.csv')
target_col = 'label'
X_raw = df.drop(columns=[target_col]).values.astype(float)
y_str = df[target_col].values          # 'Cammeo' / 'Osmancik'
y_int = (y_str == 'Osmancik').astype(int)   # Osmancik=1, Cammeo=0

X_norm, _ = utils.normalize(X_raw, X_raw)
y_cv = y_int.astype(str)               # string labels for CV helpers

print(f"Instances: {len(y_int)}  Features: {X_norm.shape[1]}")
print(f"Classes: Cammeo={int(np.sum(y_int==0))}  Osmancik={int(np.sum(y_int==1))}")


## Random Forest – Hyperparameter Sweep (ntree)

In [ ]:
feature_types = ['num'] * X_norm.shape[1]
NTREE_VALUES  = [5, 10, 20, 30, 50, 75, 100, 150]
rf_results    = {}

for nt in NTREE_VALUES:
    def model_fn(Xtr, ytr, Xte, nt=nt):
        trees = random_forest.train(Xtr, ytr, feature_types, nt)
        preds = random_forest.predict(trees, Xte)
        return preds

    acc, f1 = utils.cross_validate(model_fn, X_norm, y_cv, pos_label='1')
    rf_results[nt] = (acc, f1)
    print(f"ntree={nt:>4}  acc={acc:.4f}  f1={f1:.4f}")


In [ ]:
best_nt = max(rf_results, key=lambda n: rf_results[n][1])
print(f"Best ntree={best_nt}  acc={rf_results[best_nt][0]:.4f}  f1={rf_results[best_nt][1]:.4f}")

nts = list(rf_results.keys())
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nts, [rf_results[n][0] for n in nts], '-o', label='Accuracy')
ax.plot(nts, [rf_results[n][1] for n in nts], '-s', label='F1')
ax.set_xlabel('ntree'); ax.set_ylabel('Score')
ax.set_title('Random Forest: Accuracy & F1 vs ntree  (Rice, 10-fold CV)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/rice_rf_ntree_sweep.png', dpi=150); plt.show()


## Neural Network – Hyperparameter Sweep (architecture × lambda)

Binary output (single sigmoid neuron). Osmancik=1, Cammeo=0.

In [ ]:
Y_bin    = y_int.reshape(-1, 1).astype(float)
folds_nn = utils.stratified_k_fold(y_cv, k=10)

NN_CONFIGS = [
    ((8,),     0.0),
    ((16,),    0.0),
    ((32,),    0.0),
    ((16, 8),  0.0),
    ((16,),    0.1),
    ((16, 8),  0.1),
]

nn_results = {}
for arch, lam in NN_CONFIGS:
    accs, f1s = [], []
    for i in range(10):
        test_idx  = np.array(folds_nn[i])
        train_idx = np.array([idx for j in range(10) if j != i for idx in folds_nn[j]])

        Xtr, Xte  = X_norm[train_idx], X_norm[test_idx]
        Ytr       = Y_bin[train_idx]
        y_te_int  = y_int[test_idx]

        layer_sizes = [X_norm.shape[1]] + list(arch) + [1]
        theta, _    = neural_network.train(Xtr, Ytr, layer_sizes,
                                           lam=lam, alpha=0.3,
                                           epochs=300, batch_size=32)
        y_pred = neural_network.predict(theta, Xte)
        acc, f1 = utils.compute_metrics(y_te_int, y_pred, pos_label=1)
        accs.append(acc); f1s.append(f1)

    nn_results[(arch, lam)] = (float(np.mean(accs)), float(np.mean(f1s)))
    print(f"arch={str(arch):<12} lam={lam}  acc={np.mean(accs):.4f}  f1={np.mean(f1s):.4f}")


In [ ]:
best_nn_key  = max(nn_results, key=lambda k: nn_results[k][1])
best_nn_arch, best_nn_lam = best_nn_key
print(f"Best NN: arch={best_nn_arch}  lam={best_nn_lam}")
print(f"  acc={nn_results[best_nn_key][0]:.4f}  f1={nn_results[best_nn_key][1]:.4f}")

labels = [f"{str(a)}\nlam={l}" for a, l in nn_results]
x      = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.2, [nn_results[k][0] for k in nn_results], 0.4, label='Accuracy')
ax.bar(x + 0.2, [nn_results[k][1] for k in nn_results], 0.4, label='F1')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_title('Neural Network: Accuracy & F1 per config  (Rice, 10-fold CV)')
ax.legend(); ax.grid(alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig('results/rice_nn_config_sweep.png', dpi=150); plt.show()


### Neural Network – Learning Curve (J vs training instances)

In [ ]:
split = int(0.8 * len(y_int))
order = np.random.permutation(len(y_int))
tr_idx, te_idx = order[:split], order[split:]

Xtr_lc, Xte_lc = X_norm[tr_idx], X_norm[te_idx]
Ytr_lc = Y_bin[tr_idx]; Yte_lc = Y_bin[te_idx]

step  = max(50, split // 14)
sizes = list(range(50, split + 1, step))
if sizes[-1] != split: sizes.append(split)

lc_costs   = []
ls_best_nn = [X_norm.shape[1]] + list(best_nn_arch) + [1]

for n in sizes:
    theta_lc, _ = neural_network.train(Xtr_lc[:n], Ytr_lc[:n], ls_best_nn,
                                        lam=best_nn_lam, alpha=0.3,
                                        epochs=300, batch_size=32)
    J, _, _ = neural_network.cost(Xte_lc, Yte_lc, theta_lc, best_nn_lam)
    lc_costs.append(J)
    print(f"n={n:>5}  J={J:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sizes, lc_costs, '-o')
ax.set_xlabel('Number of training instances'); ax.set_ylabel('Cost J (test set)')
ax.set_title(f'NN Learning Curve  arch={best_nn_arch}  lam={best_nn_lam}  (Rice)')
ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/rice_nn_learning_curve.png', dpi=150); plt.show()


## Summary – Dataset 3 (Rice)

In [ ]:
print(f"{'Algorithm':<25} {'Best Hyperparameter':<22} {'Accuracy':>10} {'F1 Score':>10}")
print('-' * 72)
print(f"{'Random Forest':<25} {'ntree='+str(best_nt):<22} {rf_results[best_nt][0]:>10.4f} {rf_results[best_nt][1]:>10.4f}")
print(f"{'Neural Network':<25} {str(best_nn_arch)+' l='+str(best_nn_lam):<22} {nn_results[best_nn_key][0]:>10.4f} {nn_results[best_nn_key][1]:>10.4f}")
